# Content-Based Filtering — Baseline Prototype

Train three classifiers on directed pair features to predict `dec` (A says yes to B),
then rank wave-mates by preference score and evaluate on `match` (mutual match).

**Input:** `two_sided_pair_features.csv`  
**CV:** 3-fold wave-based split (no cross-wave leakage)  
**Models:** Logistic Regression, LightGBM, Random Forest (calibrated)

Run all cells top to bottom.

## 1. Imports and Setup

In [ ]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler

DATA_PATH = Path("..") / "data" / "two_sided_pair_features.csv"
RANDOM_STATE = 42
K_VALUES = [3, 5]
MODEL_NAMES = ["Logistic Regression", "LightGBM", "Random Forest"]

print(f"Data path: {DATA_PATH.resolve()}")

## 2. Load Data

In [ ]:
df = pd.read_csv(DATA_PATH)

feature_cols = [c for c in df.columns if c.endswith("_A") or c.endswith("_B")]

print(f"Rows: {len(df):,}")
print(f"Waves: {sorted(df['wave'].unique())}")
print(f"Features: {len(feature_cols)}")
print(f"dec=1 rate: {df['dec'].mean():.3f}")
print(f"match=1 rate: {df['match'].mean():.3f}")
df.head()

## 3. Cross-Validation Fold Definitions

In [ ]:
FOLDS = {
    1: [1, 4, 5, 17, 19, 20, 21],
    2: [3, 6, 7, 11, 13, 15, 18],
    3: [2, 8, 9, 10, 14, 16],
}

all_waves = set(df["wave"].unique())
fold_waves = set(w for waves in FOLDS.values() for w in waves)
assert all_waves == fold_waves, f"Fold waves {fold_waves} != data waves {all_waves}"

for fold_id, waves in FOLDS.items():
    print(f"Fold {fold_id}: waves {waves}")

## 4. Feature Preparation Functions

In [ ]:
def prepare_imputed_features(X_train, X_test):
    """Mean-impute using training statistics only (for LR and RF)."""
    imputer = SimpleImputer(strategy="mean")
    X_train_imputed = imputer.fit_transform(X_train)
    X_test_imputed = imputer.transform(X_test)
    return X_train_imputed, X_test_imputed


def prepare_scaled_features(X_train_imputed, X_test_imputed):
    """Standard-scale using training statistics only (for LR)."""
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imputed)
    X_test_scaled = scaler.transform(X_test_imputed)
    return X_train_scaled, X_test_scaled

## 5. Model Definitions

In [ ]:
def make_logistic_regression():
    return LogisticRegression(
        C=1.0,
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )


def make_lightgbm():
    return lgb.LGBMClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        verbose=-1,
    )


def make_random_forest():
    base_model = RandomForestClassifier(
        n_estimators=100,
        max_depth=6,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )
    return CalibratedClassifierCV(base_model, cv=3, method="isotonic")

## 6. Evaluation Metric Functions

In [ ]:
def mutual_match_at_k(ranked_list, matches, k):
    """Fraction of real mutual matches found in top-k."""
    if len(matches) == 0:
        return None
    top_k = ranked_list[:k]
    hits = sum(1 for pid in top_k if pid in matches)
    return hits / min(k, len(matches))


def ndcg_at_k(ranked_list, matches, k):
    """NDCG@k using binary relevance (match=1)."""
    if len(matches) == 0:
        return None
    top_k = ranked_list[:k]
    dcg = sum(
        1 / np.log2(i + 2)
        for i, pid in enumerate(top_k)
        if pid in matches
    )
    ideal_hits = min(k, len(matches))
    idcg = sum(1 / np.log2(i + 2) for i in range(ideal_hits))
    if idcg == 0:
        return None
    return dcg / idcg


def get_unilateral_ranking(scores_df):
    rankings = {}
    for iid, group in scores_df.groupby("iid"):
        ranked_pids = group.sort_values("score", ascending=False)["pid"].tolist()
        rankings[iid] = ranked_pids
    return rankings


def get_reciprocal_ranking(scores_df):
    score_lookup = dict(
        zip(zip(scores_df["iid"], scores_df["pid"]), scores_df["score"])
    )

    rankings = {}
    for iid, group in scores_df.groupby("iid"):
        mutual_scores = []
        for _, row in group.iterrows():
            pid = row["pid"]
            score_ab = row["score"]
            score_ba = score_lookup.get((pid, iid), None)

            if score_ba is None:
                print(f"Warning: missing reverse score for ({pid}, {iid})")
                continue

            mutual = np.sqrt(score_ab * score_ba)
            mutual_scores.append((pid, mutual))

        ranked_pids = [
            pid for pid, _ in sorted(mutual_scores, key=lambda x: x[1], reverse=True)
        ]
        rankings[iid] = ranked_pids

    return rankings


def evaluate_ranking(rankings, scores_df, k_values=None):
    if k_values is None:
        k_values = K_VALUES

    results = {f"MM@{k}": [] for k in k_values}
    results.update({f"NDCG@{k}": [] for k in k_values})
    excluded = 0
    total = 0

    for iid, ranked_list in rankings.items():
        matches = set(
            scores_df[(scores_df["iid"] == iid) & (scores_df["match"] == 1)]["pid"]
        )
        total += 1
        if len(matches) == 0:
            excluded += 1
            continue

        for k in k_values:
            mm = mutual_match_at_k(ranked_list, matches, k)
            ndcg = ndcg_at_k(ranked_list, matches, k)
            if mm is not None:
                results[f"MM@{k}"].append(mm)
            if ndcg is not None:
                results[f"NDCG@{k}"].append(ndcg)

    averages = {key: np.mean(vals) for key, vals in results.items()}
    averages["excluded"] = excluded
    averages["total"] = total
    return averages


def compute_intermediary_metrics(model, X_test, y_test_dec):
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred_class = model.predict(X_test)
    return {
        "accuracy": accuracy_score(y_test_dec, y_pred_class),
        "auc": roc_auc_score(y_test_dec, y_pred_proba),
        "logloss": log_loss(y_test_dec, y_pred_proba),
        "y_pred_proba": y_pred_proba,
    }

## 7. Main Loop — iterate over 3 folds

In [ ]:
intermediary_fold_results = {
    name: {metric: [] for metric in ["accuracy", "auc", "logloss"]}
    for name in MODEL_NAMES
}
ranking_fold_results = {
    name: {"unilateral": [], "reciprocal": []}
    for name in MODEL_NAMES
}

for fold_id, test_waves in FOLDS.items():
    print(f"\n{'=' * 60}")
    print(f"FOLD {fold_id} — test waves: {test_waves}")
    print(f"{'=' * 60}")

    # 7a. Split train/test by wave
    train_df = df[~df["wave"].isin(test_waves)].copy()
    test_df = df[df["wave"].isin(test_waves)].copy()

    train_waves = train_df["wave"].unique().tolist()
    assert len(set(train_waves) & set(test_waves)) == 0, \
        "Data leakage: test waves in training set"

    test_pairs = set(zip(test_df["iid"], test_df["pid"]))
    for a, b in test_pairs:
        assert (b, a) in test_pairs, f"Missing reverse pair ({b}, {a})"

    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]
    y_train = train_df["dec"].values
    y_test_dec = test_df["dec"].values

    # 7b. Prepare features
    X_train_imputed, X_test_imputed = prepare_imputed_features(X_train, X_test)
    X_train_scaled, X_test_scaled = prepare_scaled_features(
        X_train_imputed, X_test_imputed
    )
    X_train_lgb = X_train
    X_test_lgb = X_test

    assert not np.isnan(X_train_imputed).any(), "NaN in training features"
    assert not np.isnan(X_test_imputed).any(), "NaN in test features"

    print(f"dec=1 rate in train: {y_train.mean():.3f}")
    print(f"match=1 rate in test: {test_df['match'].mean():.3f}")
    print(f"Training rows: {len(X_train)}")
    print(f"Test rows:     {len(X_test)}")

    model_configs = {
        "Logistic Regression": (make_logistic_regression(), X_train_scaled, X_test_scaled),
        "LightGBM": (make_lightgbm(), X_train_lgb, X_test_lgb),
        "Random Forest": (make_random_forest(), X_train_imputed, X_test_imputed),
    }

    for model_name, (model, X_tr, X_te) in model_configs.items():
        print(f"\n--- {model_name} ---")

        # 7c. Train each model
        model.fit(X_tr, y_train)

        # 7d. Compute intermediary metrics (dec)
        metrics = compute_intermediary_metrics(model, X_te, y_test_dec)
        intermediary_fold_results[model_name]["accuracy"].append(metrics["accuracy"])
        intermediary_fold_results[model_name]["auc"].append(metrics["auc"])
        intermediary_fold_results[model_name]["logloss"].append(metrics["logloss"])

        print(
            f"  dec metrics — Accuracy: {metrics['accuracy']:.3f} | "
            f"AUC-ROC: {metrics['auc']:.3f} | Log-loss: {metrics['logloss']:.3f}"
        )

        # 7e. Generate preference scores
        scores_df = pd.DataFrame({
            "iid": test_df["iid"].values,
            "pid": test_df["pid"].values,
            "wave": test_df["wave"].values,
            "score": metrics["y_pred_proba"],
            "match": test_df["match"].values,
        })
        assert scores_df["score"].between(0, 1).all(), "Scores outside [0, 1]"

        # 7f. Unilateral ranking and evaluation
        uni_rankings = get_unilateral_ranking(scores_df)
        uni_eval = evaluate_ranking(uni_rankings, scores_df)
        ranking_fold_results[model_name]["unilateral"].append(uni_eval)

        print(
            f"  Unilateral  — MM@3: {uni_eval['MM@3']:.3f} | MM@5: {uni_eval['MM@5']:.3f} | "
            f"NDCG@3: {uni_eval['NDCG@3']:.3f} | NDCG@5: {uni_eval['NDCG@5']:.3f} | "
            f"excluded: {uni_eval['excluded']}/{uni_eval['total']}"
        )

        # 7g. Reciprocal re-ranking and evaluation
        rec_rankings = get_reciprocal_ranking(scores_df)
        rec_eval = evaluate_ranking(rec_rankings, scores_df)
        ranking_fold_results[model_name]["reciprocal"].append(rec_eval)

        print(
            f"  Reciprocal  — MM@3: {rec_eval['MM@3']:.3f} | MM@5: {rec_eval['MM@5']:.3f} | "
            f"NDCG@3: {rec_eval['NDCG@3']:.3f} | NDCG@5: {rec_eval['NDCG@5']:.3f} | "
            f"excluded: {rec_eval['excluded']}/{rec_eval['total']}"
        )

## 8. Aggregate and Print Final Results Table

In [ ]:
def mean_metric(fold_list, key):
    return np.mean([fold[key] for fold in fold_list])


print("\n=== Intermediary Metrics (predicting dec) ===\n")
print(f"{'Model':<22} {'Accuracy':>10} {'AUC-ROC':>10} {'Log-loss':>10}")
for model_name in MODEL_NAMES:
    res = intermediary_fold_results[model_name]
    print(
        f"{model_name:<22} "
        f"{np.mean(res['accuracy']):>10.3f} "
        f"{np.mean(res['auc']):>10.3f} "
        f"{np.mean(res['logloss']):>10.3f}"
    )

print("\n=== Per-fold Intermediary Metrics ===")
for model_name in MODEL_NAMES:
    res = intermediary_fold_results[model_name]
    print(f"\n{model_name}:")
    for fold_id in FOLDS:
        idx = fold_id - 1
        print(
            f"  Fold {fold_id}: Accuracy={res['accuracy'][idx]:.3f} | "
            f"AUC={res['auc'][idx]:.3f} | Log-loss={res['logloss'][idx]:.3f}"
        )

print("\n=== Final System Metrics (predicting match) ===")
print("Note: participants with zero matches excluded from averages")
print()
print(
    f"{'Model':<22} {'Condition':<12} {'MM@3':>8} {'MM@5':>8} "
    f"{'NDCG@3':>8} {'NDCG@5':>8}"
)
for model_name in MODEL_NAMES:
    for condition in ["unilateral", "reciprocal"]:
        fold_list = ranking_fold_results[model_name][condition]
        print(
            f"{model_name:<22} {condition.capitalize():<12} "
            f"{mean_metric(fold_list, 'MM@3'):>8.3f} "
            f"{mean_metric(fold_list, 'MM@5'):>8.3f} "
            f"{mean_metric(fold_list, 'NDCG@3'):>8.3f} "
            f"{mean_metric(fold_list, 'NDCG@5'):>8.3f}"
        )

print("\n=== Per-fold Final System Metrics ===")
for model_name in MODEL_NAMES:
    print(f"\n{model_name}:")
    for fold_id in FOLDS:
        idx = fold_id - 1
        uni = ranking_fold_results[model_name]["unilateral"][idx]
        rec = ranking_fold_results[model_name]["reciprocal"][idx]
        print(f"  Fold {fold_id} Unilateral  — "
              f"MM@3={uni['MM@3']:.3f} MM@5={uni['MM@5']:.3f} "
              f"NDCG@3={uni['NDCG@3']:.3f} NDCG@5={uni['NDCG@5']:.3f} "
              f"(excluded {uni['excluded']}/{uni['total']})")
        print(f"  Fold {fold_id} Reciprocal  — "
              f"MM@3={rec['MM@3']:.3f} MM@5={rec['MM@5']:.3f} "
              f"NDCG@3={rec['NDCG@3']:.3f} NDCG@5={rec['NDCG@5']:.3f} "
              f"(excluded {rec['excluded']}/{rec['total']})")